# Validate Equations for After Redefinitions Industry Output Estimation

Implementation of the matrix algebra used to describe the estimation of gross industry output after redefinitions for non-IO table years [in the US Methods paper](https://github.com/cornerstone-data/papers/blob/d39477c563edbe0b0e36c72f31cf1987b54cff7e/us-methods/us-methods.md). 

The implementation is a simple validation test using simplified, mock Before Redefinitions and After Redefinitions Make tables. The implementation is in numpy. It uses the Cornerstone/USEEIO nomenclature as V for Make, x for industry output, q for commodity output. This implementation limited to square matrices with identical row and column indices (same rows and column).

To be a valid example, the industry output, calculated as the row sums, must different between the two tables, where more on diagonal production is present in Va (Make After Redefinitions). But commodity output must be the same for both Make Before Redefinitions (Vb) and Va. 

Define Vb and Va where industries are the rows and commodities are the columns. But the indices are assumed identical such that row 1 industry has the col 1 primary commodity. So the diagonal is where production of primary commodity is found.

In [12]:
import numpy as np
Vb = np.array([[900.0, 100.0,0.0], [50, 450, 25],[0, 0, 75]])
Va = np.array([[925.0, 50.0, 0.0], [25, 500, 0],[0, 0, 100] ])

Calculate industry output as the row sums for each matrix, check that they are NOT identical.

In [13]:
xb = Vb.sum(axis=1) 
xa = Va.sum(axis=1)
np.array_equal(xb,xa)

False

Calculate commodity output as the column sums for each matrix, check that they are identical.

In [14]:
qb = Vb.sum(axis=0) 
qa = Va.sum(axis=0)
np.array_equal(qb,qa)

True

Now compute difference in before and after Make tables, but set diagonals to zero, so it is only differences in co-production.

In [15]:
Vstar  = Vb - Va
np.fill_diagonal(Vstar,0) # this does it in place
Vstar

array([[ 0., 50.,  0.],
       [25.,  0., 25.],
       [ 0.,  0.,  0.]])

Calculate redefinition ratios matrix, R

In [16]:
R = np.diag(1/xb) @ Vstar
R

array([[0.        , 0.05      , 0.        ],
       [0.04761905, 0.        , 0.04761905],
       [0.        , 0.        , 0.        ]])

Introduce the validation test.  Assume the target year before redefinitions output is the same as the before redefinitions output. Use the derived R matrix to calculate after redefinitions output. Check if it matches the original after redefinitions output. 

In [ ]:
xb_y = xb 
Vstar_y = np.diag(xb_y) @ R 

#Make 3x1 i vector to sum rows
i = np.array([1,1,1])

# Subtract row sums of new diffs
xa_y = xb_y - (Vstar_y @ i)

# Add col sums of new diffs
#Make 1x3 i vector to sum cols
i = np.array([[1,1,1]])

xa_y = xa_y + (i @ Vstar_y)

# Check that the new after redefinitions output equals the new after redefinitions output
# The new vector has a slightly different orientation so the check needs to be one by one
xa_y == xa


array([[ True,  True,  True]])